## Importar librerías y cargar API key de Claude

In [2]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 6.6 MB/s eta 0:00:00


In [3]:
# Importación de librerías necesarias
from anthropic import Anthropic, BadRequestError
from google.colab import userdata
import pandas as pd
import re
from tqdm import tqdm
tqdm.pandas()

# Cargar API key de Anthropic desde variables seguras de Colab
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
# Inicializar cliente de Anthropic
client = Anthropic(api_key=ANTHROPIC_API_KEY)
# Seleccionar modelo
MODEL = "claude-sonnet-4-6"

## Cargar dataset

In [4]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Cargar datasets individuales por tarea
path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
path_tara5 = path + "tara5/"

df_anx = pd.read_csv(path + "anxiety_llm.csv", sep=",")
df_dep = pd.read_csv(path + "depression_llm.csv", sep=",")
df_ed = pd.read_csv(path + "ed_llm.csv", sep=",")
df_multi = pd.read_csv(path + "multiclass_llm.csv", sep=",")

df_anx_tara5 = pd.read_csv(path_tara5 + "anxiety_llm_tara5.csv")
df_dep_tara5 = pd.read_csv(path_tara5 + "depression_llm_tara5.csv")
df_ed_tara5  = pd.read_csv(path_tara5 + "ed_llm_tara5.csv")
df_multi_tara5 = pd.read_csv(path_tara5 + "multiclass_llm_tara5.csv")

# Seleccionar tarea de clasificacion: anx | dep | ed | multi
task = "anx" # <--- Cambiar aquí

# Asignar el dataframe correspondiente a la tarea seleccionada
if task == "anx":
  df = df_anx
  df_tara5 = df_anx_tara5
elif task == "dep":
  df = df_dep
  df_tara5 = df_dep_tara5
elif task == "ed":
  df = df_ed
  df_tara5 = df_ed_tara5
elif task == "multi":
  df = df_multi
  df_tara5 = df_multi_tara5

## Función para construir el prompt

Se añade el texto a clasificar a un prompt fijo que cambio según la tarea.

In [6]:
def build_prompt(text, task):
  """ Construcción dinámica del prompt según tarea.
  Devuelve el texto completo que se enviará al modelo.
  """

  if task == "anx":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de ansiedad.

La ansiedad se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa. Estas experiencias suelen generar malestar emocional sostenido y dificultad para relajarse.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de ansiedad
0: el texto no muestra indicios de ansiedad

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "dep":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de depresión.

La depresión se asocia a un estado persistente de tristeza, apatía o vacío emocional, acompañado de una disminución del interés o la capacidad de disfrute en actividades habituales. Estas experiencias pueden incluir pensamientos negativos recurrentes, fatiga emocional y una percepción pesimista de uno mismo o del futuro.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de depresión
0: el texto no muestra indicios de depresión

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "ed":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de un trastorno de la conducta alimentaria.

Los trastornos de la conducta alimentaria se asocian a una preocupación persistente por el peso corporal, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida, el cuerpo o el control del peso. Estas preocupaciones suelen generar malestar emocional, culpa, ansiedad en torno a la alimentación y conductas de evitación o control.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de un trastorno de la conducta alimentaria
0: el texto no muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  else:
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede reflejar indicios de uno de los siguientes estados psicológicos, o no reflejar ninguno de ellos.

Ansiedad: se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa, generando malestar emocional y dificultad para relajarse.

Depresión: se asocia a un estado persistente de tristeza, apatía o vacío emocional, junto con una disminución del interés o la capacidad de disfrute en actividades habituales, pensamientos negativos recurrentes y una visión pesimista de uno mismo o del futuro.

Trastornos de la conducta alimentaria: se asocian a una preocupación persistente por el peso, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida o el cuerpo, así como malestar emocional, culpa o ansiedad en torno a la alimentación.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

0: el texto no muestra indicios de ninguno de los estados descritos
1: el texto muestra indicios de ansiedad
2: el texto muestra indicios de depresión
3: el texto muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""

## Calcular número de tokens y coste

In [ ]:
def total_tokens_dataset(df, task):
  """ Calcula el número total de tokens de entrada
  para un dataset completo, usando el tokenizador del modelo
  """
  total_tokens = 0

  for text in df["texto"]:
    # Contar tokens del prompt completo (instrucciones + texto)
    response = client.messages.count_tokens(
        model=MODEL,
        messages=[{
          "role": "user",
          "content": build_prompt(text, task)
        }]
    )
    total_tokens += response.input_tokens

  return total_tokens


# Cálculo de tokens por dataset
total_anx = total_tokens_dataset(df_anx, "anx")
print("DATASET ANSIEDAD")
print("Tokens:", total_anx)
print("Media por texto:", total_anx / len(df_anx))

total_dep = total_tokens_dataset(df_dep, "dep")
print("DATASET DEPRESIÓN")
print("Tokens:", total_dep)
print("Media por texto:", total_dep / len(df_dep))

total_ed = total_tokens_dataset(df_ed, "ed")
print("DATASET ED")
print("Tokens:", total_ed)
print("Media por texto:", total_ed / len(df_ed))

total_multi = total_tokens_dataset(df_multi, "multi")
print("DATASET MULTICLASE")
print("Tokens:", total_multi)
print("Media por texto:", total_multi / len(df_multi))

# Suma total de tokens necesarios para todo el experimento
print(f"Total de tokens: {total_anx + total_dep + total_ed + total_multi}")

DATASET ANSIEDAD
Tokens: 445813
Media por texto: 891.626
DATASET DEPRESIÓN
Tokens: 372878
Media por texto: 747.2505010020041
DATASET DEPRESIÓN
Tokens: 246501
Media por texto: 735.823880597015
DATASET MULTICLASE
Tokens: 1065192
Media por texto: 798.4947526236881
Total de tokens: 2130384


## Clasificación zero-shot con Claude

In [7]:
import anthropic

def classify_text(text, task):
    """Función de clasificación usando Claude.
    Devuelve:
    - int (0,1,2,3) si hay predicción válida
    - None si el modelo bloquea o no devuelve texto
    """
    try:
        response = client.messages.create(
            model=MODEL,
            messages=[{
                "role": "user",
                "content": build_prompt(text, task)
            }],
            temperature=0,  # Generación determinista
            max_tokens=5    # Limitar longitud de salida
        )

        # En Claude la respuesta es una lista de bloques de contenido
        if not response.content:
            return None

        text_response = response.content[0].text

        # Extraer número de clase mediante expresión regular
        match = re.search(r"[0-3]", text_response)
        return int(match.group()) if match else None

    except anthropic.BadRequestError:
        # Claude lanza excepción cuando bloquea el contenido
        # (equivalente al PROHIBITED_CONTENT de Gemini)
        return None

In [ ]:
# Aplicar clasificación al dataset seleccionado
# Se almacena la predicción en una nueva columna "pred"
print(f"Iniciando predicción para el dataset: {task} | Total textos: {len(df)}")
df["pred"] = df["texto"].progress_apply(lambda x: classify_text(x, task))

Iniciando predicción para el dataset: ed | Total textos: 335


100%|██████████| 335/335 [10:48<00:00,  1.94s/it]


### Comprobación de predicciones

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Reemplazar None por una clase inválida para que cuente como error
pred_labels = df["pred"].fillna(-1)

# Si es clasificación binaria
if task in ["anx", "dep", "ed"]:
    # Columna con la etiqueta real
    true_labels = df["bs"]
    f1 = f1_score(true_labels, pred_labels, average="binary")
else:
    # Columna con la etiqueta real
    true_labels = df["label_mc"]
    # Clasificación multiclase
    f1 = f1_score(true_labels, pred_labels, average="macro")

acc = accuracy_score(true_labels, pred_labels)

print("F1-score:", round(f1, 4))
print("Acc:", round(acc, 4))

F1-score: 0.9283
Acc: 0.9373


In [ ]:
# Número de bloqueos
num_blocked = df["pred"].isna().sum()
block_rate = num_blocked / len(df)

print("Bloqueos:", num_blocked)
print("Tasa de bloqueo:", round(block_rate, 4))

Bloqueos: 0
Tasa de bloqueo: 0.0


### Almacenar predicciones

In [ ]:
output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}.csv"
)
df.to_csv(output_path, index=False)

## Clasificación múltiple para Tara@5

Para analizar la estabilidad de las predicciones, se aplicará TARa@5 sobre un subconjunto estratificado del dataset. Cada ejemplo se evalúa cinco veces con el mismo modelo, reutilizando la primera predicción y generando cuatro ejecuciones adicionales, lo que permite medir el grado de consistencia de las salidas.

In [8]:
print(f"Iniciando TARa@5 para dataset: {task} | Textos: {len(df_tara5)}")

output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}_tara5.csv"
)

# Aplicar la misma función de clasificación 4 veces más
# Se almacena cada predicción en una nueva columna "pred_run_{num_run}"
for run in range(1, 5):
    col_name = f"pred_run{run+1}"

    if col_name in df_tara5.columns:
        print(f"{col_name} ya existe, se omite")
        continue

    print(f"\nEjecutando pred_run{run+1}...")

    df_tara5[col_name] = df_tara5["texto"].progress_apply(lambda x: classify_text(x, task))

    # Guardado incremental
    df_tara5.to_csv(output_path, index=False)

    print(f"Run {run+1} guardada correctamente")

Iniciando TARa@5 para dataset: anx | Textos: 100

Ejecutando pred_run2...


100%|██████████| 100/100 [02:57<00:00,  1.77s/it]


Run 2 guardada correctamente

Ejecutando pred_run3...


100%|██████████| 100/100 [03:31<00:00,  2.11s/it]


Run 3 guardada correctamente

Ejecutando pred_run4...


100%|██████████| 100/100 [03:46<00:00,  2.27s/it]


Run 4 guardada correctamente

Ejecutando pred_run5...


100%|██████████| 100/100 [03:41<00:00,  2.22s/it]

Run 5 guardada correctamente
